In [ ]:
import pandas as pd
import ast
import random
import string
import json
from sentence_transformers import SentenceTransformer
import requests


/home/mortadha/Pilot/Benchmark/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2643.85it/s]


In [ ]:
def compute_similarity(sim_model, sentence1, sentence2):  
  # Compute embeddings for both lists
    embeddings1 = sim_model.encode(sentence1)
    embeddings2 = sim_model.encode(sentence2)

    # Compute cosine similarities
    similarity= sim_model.similarity(embeddings1, embeddings2)[0]
    return similarity


In [ ]:
def anonymize_speakers(speakers):
    def generate_id():
        letters = ''.join(random.choices(string.ascii_uppercase, k=3))
        numbers = ''.join(random.choices(string.digits, k=2))
        return letters + numbers
    speakers = ast.literal_eval(speakers) if isinstance(speakers, str) else speakers
    unique_speakers = list(set(speakers))
    mapping = {speaker: generate_id() for i,speaker in enumerate(unique_speakers)}
    return mapping


In [95]:
def generate_anonymized_dialogue(row,format = "{speaker}:{turn}" ,chunked=False, max_chunk_size=10):
    """
    Generate an anonymized dialogue based on the input row.
    If chunked is True, the dialogue will be split into chunks of a RANDOM size with a minimum of 2 turns and a maximum max_chunk_size turns. Choose a random chunk size for each chunk between 2 and max_chunk_size, so that the chunks are not always the same size.
    Use the max_chunk_size as a starting point to get the following chunk and return a list of chunks containing len(turns) // max_chunk_size chunks. If the last chunk has less than 2 turns, merge it with the previous chunk.
    Every chunk should be a string of turns formatted based on the input format, where {speaker} is replaced with the anonymized speaker ID and {turn} is replaced with the turn text followed by a newline character.
    """
    speakers = ast.literal_eval(row["speakers"]) if isinstance(row["speakers"], str) else row["speakers"]
    turns = ast.literal_eval(row["turns"]) if isinstance(row["turns"], str) else row["turns"]
    speaker_mapping = row["speaker_mapping"]
    dialogue = []
    for speaker, turn in zip(speakers, turns):
        anonymized_speaker = speaker_mapping[speaker]
        dialogue.append(format.format(speaker=anonymized_speaker, turn=turn))
    dialogue = "\n".join(dialogue)
    if chunked:
        chunks = []
        i = 0
        while i < len(turns):
            chunk_size = random.randint(2, max_chunk_size)
            chunk_turns = turns[i:i+chunk_size]
            chunk_speakers = speakers[i:i+chunk_size]
            chunk_dialogue = []
            for speaker, turn in zip(chunk_speakers, chunk_turns):
                anonymized_speaker = speaker_mapping[speaker]
                chunk_dialogue.append(format.format(speaker=anonymized_speaker, turn=turn))
            chunks.append("\n".join(chunk_dialogue))
            i += chunk_size
        # Merge last chunk if it has less than 2 turns
        if len(chunks) > 1 and len(chunks[-1].split("\n")) < 2:
            chunks[-2] += "\n" + chunks[-1]
            chunks.pop()
        return chunks
    return [dialogue]


In [ ]:
def evaluate(data,col,generate,sim_):
    """
    Evaluate the LLM's ability to identify the learning context of a dialogue based on the generated anonymized dialogue.
    """
    with open("label_map.json", "r") as f:
        label_map = json.load(f)
    with open("prompts.json", "r") as f:
        prompts = json.load(f)
    
    testset = []
    for value in label_map[col].values():
        if data[data[col]==value].shape[0] > 100:
            test = data[data[col]==value].sample(100, random_state=42)
            test["dialogue"]= test.apply(lambda x: generate_anonymized_dialogue(x)[0],axis=1)
            testset.append(test[["dialogue",col]])
        else:
            examples = []
            test = data[data[col]==value]
            for ind,row in test.iterrows():
                dialogues = generate_anonymized_dialogue(row,chunked=True,max_chunk_size=20)
                for dialogue in dialogues:
                    examples.append({"dialogue":dialogue,col:row[col]})
            testset.append(pd.DataFrame(examples).sample(100, random_state=42))
    testset = pd.concat(testset,ignore_index=True).sample(frac=1)
    testset["generated"] = testset["dialogue"].apply(lambda x: generate(random.choice(list(prompts[col].values())).format(INPUT_DIALOGUE=x)))
    if col in label_map.keys():
        score = (testset["generated"].apply(lambda x: label_map[col][x]) == testset[col]).mean()
    else:
        score = testset.apply(lambda x: compute_similarity(x["generated"], x[col]), axis=1).mean()
    return score



In [ ]:
def generate_open_ai(message,url,key,model):
    
    headers = {
    'Authorization': f'Bearer {key}',
    'Content-Type': 'application/json'
    }
    data = {
    "model": model,
    "messages":[{"role": "user", "content": message}],
    "stream": False
    }
    response = requests.post(url, headers=headers, json=data)
    return response.json()["message"]["content"]

In [ ]:
"""
main function to evaluate the LLM abilities.
It takes as terminal arguments the following:
- the column to evaluate from the following      ["learning_context", "comm_modality","agent_config", "subject", "edu_level"] if not specified, it evaluates all the columns.
- the backend to use for evaluation for now only openai and unsloth
- api key (only required for openai)
- url(only required for openai)
- model name
"""
if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Evaluate LLM abilities")
    parser.add_argument("--column", type=str, help="The column to evaluate")
    parser.add_argument("--backend", type=str, help="The backend to use for evaluation", default="openai")
    parser.add_argument("--api_key", type=str, help="The API key for openai", default=None)
    parser.add_argument("--url", type=str, help="The URL for openai", default=None)
    parser.add_argument("--model", type=str, help="The model name for openai", default=None)
    args = parser.parse_args()
    
    sim_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    data = pd.read_csv("/home/mortadha/Pilot/Benchmark/data/processed_datasets.csv")
    data["speaker_mapping"] = data["speakers"].apply(anonymize_speakers)
    if args.backend == "openai":
        generate = lambda message: generate_open_ai(message,args.url,args.api_key,args.model)
    else:
        raise NotImplementedError("Only openai backend is supported for now")
    if args.column:
        score = evaluate(data,args.column,generate,sim_model)
        print(f"Score for {args.column}: {score}")
    else:
        columns = ["learning_context", "comm_modality","agent_config", "subject", "edu_level"]
        scores = {}
        for col in columns:
            score = evaluate(data,col,generate,sim_model)
            scores[col] = score
            print(f"Score for {col}: {score}")

In [122]:
data = pd.read_csv("/home/mortadha/Pilot/Benchmark/data/processed_datasets.csv")
data["speaker_mapping"] = data["speakers"].apply(anonymize_speakers)

In [123]:
generate = lambda x: random.choice(["A","B","C"])
col = "edu_level"
with open("label_map.json", "r") as f:
    label_map = json.load(f)
with open("prompts.json", "r") as f:
    prompts = json.load(f)
print(data.columns)
testset = []
for value in label_map[col].values():
    if data[data[col]==value].shape[0] > 100:
        test = data[data[col]==value].sample(100, random_state=42)
        test["dialogue"]= test.apply(lambda x: generate_anonymized_dialogue(x)[0],axis=1)
        testset.append(test[["dialogue",col]])
    else:
        """examples = []
        test = data[data[col]==value]
        for ind,row in test.iterrows():
            dialogues = generate_anonymized_dialogue(row,chunked=True,max_chunk_size=20)
            for dialogue in dialogues:
                examples.append({"dialogue":dialogue,col:row[col]})
        testset.append(pd.DataFrame(examples).sample(100, random_state=42))"""
        continue
testset = pd.concat(testset,ignore_index=True).sample(frac=1)
testset["generated"] = testset["dialogue"].apply(lambda x: generate(random.choice(list(prompts[col].values())).format(INPUT_DIALOGUE=x)))


Index(['speakers', 'turns', 'dialogic_acts', 'edu_level', 'learning_context',
       'comm_modality', 'agent_config', 'subject', 'language', 'dataset',
       'extra', 'speaker_mapping'],
      dtype='object')


In [124]:
testset

,dialogue,edu_level,generated
54,"WXM44:HI Cody, can you tell me where you got 4...",middle,B
60,JWV20:Hi Samantha!!\nOXU44:Hello again !!\nJWV...,middle,C
97,PCW54:How long does it take the rabbit to run ...,middle,A
30,BUT23:where did you get 1.5 km?\nXCV10:I got 1...,middle,C
92,"ZUC14:you have some good idea\nDLO21:Yes, I un...",middle,B
...,...,...,...
26,QVI49:Hello Liana 😁\nQVI49:Can I help you with...,middle,A
90,"IRT50:Hi Nathaniel!\nIRT50:So, you'll need to ...",middle,C
21,"ROI82:Alright, here we go.\nROI82:Um, so our l...",middle,A
72,SWN27:Hi! How can I help?\nAXA31:I’m stuck\nSW...,middle,B


In [2]:
import pandas as pd
data = pd.read_csv("/home/mortadha/Pilot/Benchmark/data/processed_datasets.csv")
data

,speakers,turns,dialogic_acts,edu_level,learning_context,comm_modality,agent_config,subject,language,dataset,extra
0,"['teacher', 'student', 'teacher', 'student', '...","['Okay. Math should be out. Everything else,...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN
1,"['teacher', 'student', 'teacher', 'student', '...","['Red. It’s red, okay. Okay. Boys and girls...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN
2,"['teacher', 'student', 'teacher', 'student', '...",['Okay. This afternoon we’re going to be doin...,"[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN
3,"['teacher', 'multiple students', 'teacher', 's...","['Okay. Okay, good afternoon boys and girls.'...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN
4,"['teacher', 'student', 'teacher', 'student', '...",['Perfect. Raise your what? I put your bag o...,"[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN
...,...,...,...,...,...,...,...,...,...,...,...
7317,"[4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 2, 1, 1, 1, ...","['Callibrate my cameras real quick', 'So the t...","[[], [], [], [], ['CONST_EstablishesCG_Confirm...",university,collaborative work,in-person,Human-Human,"physics, problem solving",english,weighttasks,NaN
7318,"[4, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 2, 3, 1, 1, ...","[""Let me Calibrate my cameras real quick Oh I'...","[[], [], ['MAINTAIN_FulfillsR_Apologizes'], []...",university,collaborative work,in-person,Human-Human,"physics, problem solving",english,weighttasks,NaN
7319,"[2, 2, 3, 3, 3, 1, 3, 1, 3, 2, 3, 3, 1, 2, 3, ...","['Or is it', 'Yeah green is for twenty', 'Ok s...","[['NEG_MonitorsE_Results'], [], [], ['CONST_Sh...",university,collaborative work,in-person,Human-Human,"physics, problem solving",english,weighttasks,NaN
7320,"[4, 4, 4, 1, 2, 4, 4, 4, 4, 4, 4, 4, 3, 2, 3, ...","[""Ok I'm going to callibrate my cameras"", ""I'm...","[[], [], [], [], ['CONST_EstablishesCG_Interru...",university,collaborative work,in-person,Human-Human,"physics, problem solving",english,weighttasks,NaN


In [ ]:
import random
import string
def anonymize_speakers(speakers):
    def generate_id():
        letters = ''.join(random.choices(string.ascii_uppercase, k=3))
        numbers = ''.join(random.choices(string.digits, k=2))
        return letters + numbers
    speakers = ast.literal_eval(speakers) if isinstance(speakers, str) else speakers
    unique_speakers = list(set(speakers))
    mapping = {speaker: generate_id() for i,speaker in enumerate(unique_speakers)}
    return mapping

def generate_anonymized_dialogue(row,format = "{speaker}:{turn}" ,chunked=False, max_chunk_size=10):
    """
    Generate an anonymized dialogue based on the input row.
    If chunked is True, the dialogue will be split into chunks of a RANDOM size with a minimum of 2 turns and a maximum max_chunk_size turns. Choose a random chunk size for each chunk between 2 and max_chunk_size, so that the chunks are not always the same size.
    Use the max_chunk_size as a starting point to get the following chunk and return a list of chunks containing len(turns) // max_chunk_size chunks. If the last chunk has less than 2 turns, merge it with the previous chunk.
    Every chunk should be a string of turns formatted based on the input format, where {speaker} is replaced with the anonymized speaker ID and {turn} is replaced with the turn text followed by a newline character.
    """
    
    speakers = ast.literal_eval(row["speakers"]) if isinstance(row["speakers"], str) else row["speakers"] 
    try:
        turns = ast.literal_eval(row["turns"]) if isinstance(row["turns"], str) else row["turns"]
    except:
        return [""]
    speaker_mapping = row["speaker_mapping"]
    dialogue = []
    for speaker, turn in zip(speakers, turns):
        anonymized_speaker = speaker_mapping[speaker]
        dialogue.append(format.format(speaker=anonymized_speaker, turn=turn))
    dialogue = "\n".join(dialogue)
    if chunked:
        chunks = []
        i = 0
        while i < len(turns):
            chunk_size = random.randint(min(4,max_chunk_size), max_chunk_size)
            chunk_turns = turns[i:i+chunk_size]
            chunk_speakers = speakers[i:i+chunk_size]
            chunk_dialogue = []
            for speaker, turn in zip(chunk_speakers, chunk_turns):
                anonymized_speaker = speaker_mapping[speaker]
                chunk_dialogue.append(format.format(speaker=anonymized_speaker, turn=turn))
            chunks.append("\n".join(chunk_dialogue))
            i += chunk_size
        # Merge last chunk if it has less than 2 turns
        if len(chunks) > 1 and len(chunks[-1].split("\n")) < 4:
            chunks[-2] += "\n" + chunks[-1]
            chunks.pop()
        return chunks
    return [dialogue]


data["speakers"] = data["speakers"].apply(lambda speakers: ast.literal_eval(speakers) if isinstance(speakers, str) else speakers)
data["speaker_mapping"] = data["speakers"].apply(lambda x: anonymize_speakers(x,seed=RD))
data = data[data["speakers"].apply(lambda speakers: "tutor" in set([str(x).lower() for x in speakers]) or "teacher" in set([str(x).lower() for x in speakers]) or "t" in set([str(x).lower() for x in speakers]))]
data["teacher"] = data["speaker_mapping"].apply(lambda x: x.get("teacher",x.get("Teacher",x.get("tutor",x.get("Tutor",x.get("t",x.get("T",None)))))))
data["dialogue"] = data.apply(lambda x: random.choice([dialog if x["teacher"] in dialog else "" for dialog in generate_anonymized_dialogue(x,chunked=True,max_chunk_size=len(x["speakers"])) ],seed = RD),axis=1)
data = data[data.apply(lambda x: x["teacher"] in x["dialogue"],axis=1)]
data


,speakers,turns,dialogic_acts,edu_level,learning_context,comm_modality,agent_config,subject,language,dataset,extra,speaker_mapping,dialogue,teacher
0,"[teacher, student, teacher, student, teacher, ...","['Okay. Math should be out. Everything else,...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN,"{'student': 'YLV73', 'teacher': 'TFN92', 'mult...","YLV73:8.\nTFN92:8 apples in each row, or –?\nY...",TFN92
1,"[teacher, student, teacher, student, teacher, ...","['Red. It’s red, okay. Okay. Boys and girls...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN,"{'student': 'ZBV03', 'teacher': 'PHZ29', 'mult...",PHZ29:So you saw that you got two and four is ...,PHZ29
2,"[teacher, student, teacher, student, teacher, ...",['Okay. This afternoon we’re going to be doin...,"[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN,"{'student': 'LFJ30', 'teacher': 'AEL42', 'mult...",AEL42:Okay. This afternoon we’re going to be ...,AEL42
3,"[teacher, multiple students, teacher, student,...","['Okay. Okay, good afternoon boys and girls.'...","[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN,"{'student': 'RHM14', 'teacher': 'VXC29', 'mult...",CNX29:Yes.\nVXC29:Okay. Stop. I will come ba...,VXC29
4,"[teacher, student, teacher, student, teacher, ...",['Perfect. Raise your what? I put your bag o...,"[['None'], ['None'], ['None'], ['None'], ['Non...",elementary,classroom,in-person,Human-Human,maths,english,ncte,NaN,"{'student': 'AJE25', 'teacher': 'DXE16', 'mult...",DXE16:Perfect. Raise your what? I put your b...,DXE16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7307,"[Teacher, Student, Teacher, Student, Teacher, ...","['Hi Luca, can you tell me how you got your an...","['generic', 'None', 'probing', 'None', 'focus'...",middle,tutoring,synchronous chat,Human-AI,maths,english,mathdial,The student forgot to add the cost of insuranc...,"{'Student': 'IQO12', 'Teacher': 'IOU85'}","IOU85:Hi Luca, can you tell me how you got you...",IOU85
7308,"[Teacher, Student, Teacher, Student, Teacher, ...","['Hi , could you please walk me through your s...","['generic', 'None', 'focus', 'None', 'focus', ...",middle,tutoring,synchronous chat,Human-AI,maths,english,mathdial,Conceptual error in third step. calculated tim...,"{'Student': 'HPF86', 'Teacher': 'NTE27'}","NTE27:Hi , could you please walk me through yo...",NTE27
7309,"[Teacher, Student, Teacher, Teacher, Teacher, ...","['Hi, could you please walk me through your so...","['generic', 'None', 'focus', 'focus', 'focus',...",middle,tutoring,synchronous chat,Human-AI,maths,english,mathdial,Repeated subtraction by deducting the number o...,"{'Student': 'EZU98', 'Teacher': 'YIE91'}","YIE91:Hi, could you please walk me through you...",YIE91
7310,"[Teacher, Student, Teacher, Student, Teacher, ...","['Hi, could you please walk me through your so...","['generic', 'None', 'focus', 'None', 'focus', ...",middle,tutoring,synchronous chat,Human-AI,maths,english,mathdial,NaN,"{'Student': 'COV18', 'Teacher': 'SUJ16'}","SUJ16:Hi, could you please walk me through you...",SUJ16
